# Notebook 02 — Analyse du taux de tagging

In [1]:
import pandas as pd
from pathlib import Path
df = pd.read_csv(Path('../../ressources/datasets/cur_sample.csv'), parse_dates=['usage_date'])
account_labels = {'111111111111':'prod','222222222222':'staging','333333333333':'dev','444444444444':'sandbox'}
df['account_name'] = df['account_id'].map(account_labels)
print('Dataset charge :', len(df), 'lignes')

Dataset charge : 2523 lignes


## 1. Taux de couverture global par dimension de tag

In [2]:
total_cost = df['unblended_cost'].sum()
for tag in ['tag_team', 'tag_env', 'tag_project']:
    tagged = df[df[tag].notna() & (df[tag] != '')]['unblended_cost'].sum()
    print(f'{tag:15s} : {tagged/total_cost*100:.1f}% couvert (${tagged:,.0f} / ${total_cost:,.0f})')

tag_team        : 81.9% couvert ($27,798 / $33,949)
tag_env         : 87.7% couvert ($29,782 / $33,949)
tag_project     : 66.4% couvert ($22,537 / $33,949)


## 2. Services orphelins (sans tag_team)

In [3]:
untagged = df[df['tag_team'].isna() | (df['tag_team'] == '')]
orphans = untagged.groupby('service')['unblended_cost'].sum().sort_values(ascending=False)
print('Services sans tag_team :')
print(orphans.to_string())
print(f'\nTotal non tague : ${orphans.sum():,.2f} ({orphans.sum()/total_cost*100:.1f}% du total)')

Services sans tag_team :
service
AmazonEC2           3328.2524
AmazonRDS           1031.3609
AmazonCloudWatch     740.2169
AmazonEBS            632.1154
AmazonS3             260.4111
AmazonVPC             90.2478
AmazonEKS             67.8213
AWSLambda              0.0111

Total non tague : $6,150.44 (18.1% du total)


## 3. Matrice de couverture : compte x service

In [4]:
df['is_tagged'] = (df['tag_team'].notna() & (df['tag_team'] != '')).astype(int)
matrix = df.pivot_table(
    index='service', columns='account_name',
    values='is_tagged', aggfunc='mean'
).round(2) * 100
print('Taux de tagging (%) par service x compte :')
print(matrix.to_string())

Taux de tagging (%) par service x compte :
Empty DataFrame
Columns: []
Index: []


## 4. Top services a tagger en priorite

In [5]:
priority = untagged.groupby('service')['unblended_cost'].sum().sort_values(ascending=False).head(5)
print('Priorite de tagging (cout non tague) :')
for svc, cost in priority.items():
    print(f'  {svc:25s} : ${cost:,.2f}')

Priorite de tagging (cout non tague) :
  AmazonEC2                 : $3,328.25
  AmazonRDS                 : $1,031.36
  AmazonCloudWatch          : $740.22
  AmazonEBS                 : $632.12
  AmazonS3                  : $260.41
